## Double Header Data


In [1]:
import re
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import requests
from datetime import date
from collections import OrderedDict
from IPython.display import display, HTML
from pathlib import Path

## ESPN

In [2]:
def espn_doubleheaders_textparse(url: str) -> pd.DataFrame:
    """
    Parse an ESPN MLB doubleheaders page into a DataFrame with columns:
    DATE | TEAMS | GAME 1 | GAME 2

    Args:
        url (str): full ESPN URL, e.g. "https://www.espn.com/mlb/stats/doubleheaders/_/year/2025"
    """
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
    resp.raise_for_status()

    # Pull visible text (works even if it's an ARIA/virtual table)
    soup = BeautifulSoup(resp.text, "lxml")
    txt = soup.get_text("\n", strip=True)

    # Regex: Month Day | Teams | Game1 | Game2
    pat = re.compile(
        r"([A-Z][a-z]+ \d{1,2})\s+(.+?)\s+([A-Z]{2,3},\s*\d+-\d+|Postponed)\s+([A-Z]{2,3},\s*\d+-\d+|Postponed)",
        flags=re.MULTILINE
    )

    rows = []
    for m in pat.finditer(txt):
        date, teams, g1, g2 = m.groups()
        rows.append({"DATE": date, "TEAMS": teams, "GAME 1": g1, "GAME 2": g2})

    if not rows:
        raise RuntimeError("Could not parse any doubleheader rows from ESPN text.")

    df = pd.DataFrame(rows, columns=["DATE", "TEAMS", "GAME 1", "GAME 2"])
    return df.reset_index(drop=True)

def normalize_doubleheader_dates(df: pd.DataFrame, year: int = 2025) -> pd.DataFrame:
    out = df.copy()
    out["DATE"] = (out["DATE"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
                   + f" {year}")
    out["DATE"] = pd.to_datetime(out["DATE"], format="%B %d %Y", errors="coerce").dt.strftime("%Y-%m-%d")
    return out

def drop_postponed_doubleheaders(df):
    """
    Remove rows from a double-header DataFrame where either GAME 1 or GAME 2
    is postponed.

    Assumes columns 'GAME 1' and 'GAME 2' exist.

    Parameters
    ----------
    df : pd.DataFrame
        Double-header DataFrame with columns 'GAME 1' and 'GAME 2'.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame with postponed double headers removed.
    """
    # Convert to string in case of NaNs or non-string entries
    game1_postponed = df["GAME 1"].astype(str).str.contains("Postponed", case=False, na=False)
    game2_postponed = df["GAME 2"].astype(str).str.contains("Postponed", case=False, na=False)

    # Keep only rows where neither game is postponed
    mask_keep = ~(game1_postponed | game2_postponed)
    return df.loc[mask_keep].reset_index(drop=True)


### 2025

In [3]:
url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2025"
dh_25 = espn_doubleheaders_textparse(url)
dh_25 = normalize_doubleheader_dates(dh_25, year=2025)
dh_25 = drop_postponed_doubleheaders(dh_25)
dh_25.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2025-04-06,St. Louis at Boston,"BOS,\n5-4","BOS,\n18-7"
1,2025-04-20,Washington at Colorado,"COL,\n3-1","WSH,\n3-2"
2,2025-04-24,Colorado at Kansas City,"KC,\n7-4","KC,\n6-2"
3,2025-04-26,Boston at Cleveland,"CLE,\n5-4","BOS,\n7-3"
4,2025-04-26,Baltimore at Detroit,"DET,\n4-3","DET,\n6-2"
5,2025-04-27,Toronto at NY Yankees,"NYY,\n5-1","NYY,\n11-2"
6,2025-04-30,St. Louis at Cincinnati,"STL,\n9-1","STL,\n6-0"
7,2025-05-04,NY Mets at St. Louis,"STL,\n6-5","STL,\n5-4"
8,2025-05-06,Cleveland at Washington,"WSH,\n10-9","CLE,\n9-1"
9,2025-05-08,Detroit at Colorado,"DET,\n10-2","DET,\n11-1"


In [4]:
url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2024"
dh_24 = espn_doubleheaders_textparse(url)
dh_24 = normalize_doubleheader_dates(dh_24, year=2024)
dh_24 = drop_postponed_doubleheaders(dh_24)
dh_24.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2024-04-04,Detroit at NY Mets,"NYM,\n2-1","DET,\n6-3"
1,2024-04-13,NY Yankees at Cleveland,"NYY,\n3-2","NYY,\n8-2"
2,2024-04-13,Minnesota at Detroit,"MIN,\n4-1","MIN,\n11-5"
3,2024-04-17,Kansas City at Chicago Sox,"KC,\n4-2","CHW,\n2-1"
4,2024-04-20,Miami at Chicago Cubs,"CHC,\n5-3","MIA,\n3-2"
5,2024-04-21,Seattle at Colorado,"SEA,\n10-2","COL,\n2-1"
6,2024-04-30,St. Louis at Detroit,"DET,\n11-6","STL,\n2-1"
7,2024-05-08,Texas at Athletics,"ATH,\n9-4","TEX,\n12-11"
8,2024-05-14,Washington at Chicago Sox,"WSH,\n6-3","CHW,\n4-0"
9,2024-05-20,San Diego at Atlanta,"ATL,\n3-0","SD,\n6-5"


### 2023

In [5]:
url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2023"
dh_23 = espn_doubleheaders_textparse(url)
dh_23 = normalize_doubleheader_dates(dh_23, year=2023)
dh_23 = drop_postponed_doubleheaders(dh_23)
dh_23.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2023-04-18,Philadelphia at Chicago Sox,"CHW,\n3-0","PHI,\n7-4"
1,2023-04-18,Cleveland at Detroit,"DET,\n4-3","DET,\n1-0"
2,2023-04-22,Miami at Cleveland,"MIA,\n3-2","MIA,\n6-1"
3,2023-04-29,Baltimore at Detroit,"BAL,\n6-4","DET,\n7-4"
4,2023-04-29,Pittsburgh at Washington,"PIT,\n16-1","PIT,\n6-3"
5,2023-05-01,Atlanta at NY Mets,"NYM,\n5-3","ATL,\n9-8"
6,2023-05-03,NY Mets at Detroit,"DET,\n8-1","DET,\n6-5"
7,2023-05-21,Cleveland at NY Mets,"NYM,\n2-1","NYM,\n5-4"
8,2023-06-03,Tampa Bay at Boston,"TB,\n4-2","BOS,\n8-5"
9,2023-06-08,Chicago Sox at NY Yankees,"CHW,\n6-5","NYY,\n3-0"


### 2022

In [6]:
url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2022"
dh_22 = espn_doubleheaders_textparse(url)
dh_22 = normalize_doubleheader_dates(dh_22, year=2022)
dh_22.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2022-04-19,Arizona at Washington,"WSH,\n6-1","WSH,\n1-0"
1,2022-04-19,San Francisco at NY Mets,"NYM,\n5-4","NYM,\n3-1"
2,2022-04-20,Chicago Sox at Cleveland,"CLE,\n11-1","CLE,\n2-1"
3,2022-04-23,Colorado at Detroit,"DET,\n13-0","COL,\n3-2"
4,2022-05-03,Atlanta at NY Mets,"NYM,\n5-4","NYM,\n3-0"
5,2022-05-04,San Diego at Cleveland,"SD,\n5-4","CLE,\n6-5"
6,2022-05-04,Pittsburgh at Detroit,"DET,\n3-2","PIT,\n7-2"
7,2022-05-07,Toronto at Cleveland,"TOR,\n8-3","CLE,\n8-2"
8,2022-05-07,LA Dodgers at Chicago Cubs,"LAD,\n7-0","LAD,\n6-2"
9,2022-05-07,Pittsburgh at Cincinnati,"CIN,\n9-2","PIT,\n8-5"


### Abbreviation

In [7]:
# Short-name → abbreviation map (your scheme)
team_to_abbr_short = {
    "Arizona": "AZ", "Athletics": "ATH", "Atlanta": "ATL", "Baltimore": "BAL",
    "Boston": "BOS", "Chicago Cubs": "CHC", "Chicago Sox": "CWS", "Cincinnati": "CIN",
    "Cleveland": "CLE", "Colorado": "COL", "Detroit": "DET", "Houston": "HOU",
    "Kansas City": "KC", "LA Angels": "LAA", "LA Dodgers": "LAD", "Miami": "MIA",
    "Milwaukee": "MIL", "Minnesota": "MIN", "NY Mets": "NYM", "NY Yankees": "NYY",
    "Philadelphia": "PHI", "Pittsburgh": "PIT", "San Diego": "SD", "San Francisco": "SF",
    "Seattle": "SEA", "St. Louis": "STL", "Tampa Bay": "TB", "Texas": "TEX",
    "Toronto": "TOR", "Washington": "WSH",
}

def _normalize_team_label(s: str) -> str:
    if s is None: return None
    s = str(s).strip()
    rules = {
        r"(?i)^chicago\s+white\s+sox$": "Chicago Sox",
        r"(?i)^chi(?:\.|cago)?\s*(white\s*)?sox$": "Chicago Sox",
        r"(?i)^chi(?:\.|cago)?\s*cubs$": "Chicago Cubs",
        r"(?i)^(los\s+angeles\s+angels|la\s+angels)$": "LA Angels",
        r"(?i)^(los\s+angeles\s+dodgers|la\s+dodgers)$": "LA Dodgers",
        r"(?i)^(new\s+york\s+yankees)$": "NY Yankees",
        r"(?i)^(new\s+york\s+mets)$": "NY Mets",
        r"(?i)^(cleveland\s+guardians)$": "Cleveland",
        r"(?i)^(arizona\s+diamondbacks)$": "Arizona",
        r"(?i)^(tampa\s+bay\s+rays)$": "Tampa Bay",
        r"(?i)^(oakland\s+athletics|a's|athletic[s]?)$": "Athletics",
        r"(?i)^(st\.?\s*louis\s*cardinals)$": "St. Louis",
        r"(?i)^(san\s+francisco\s+giants)$": "San Francisco",
        r"(?i)^(san\s+diego\s+padres)$": "San Diego",
        r"(?i)^(seattle\s+mariners)$": "Seattle",
        r"(?i)^(texas\s+rangers)$": "Texas",
        r"(?i)^(toronto\s+blue\s+jays)$": "Toronto",
        r"(?i)^(washington\s+nationals)$": "Washington",
        r"(?i)^(philadelphia\s+phillies)$": "Philadelphia",
        r"(?i)^(pittsburgh\s+pirates)$": "Pittsburgh",
        r"(?i)^(cincinnati\s+reds)$": "Cincinnati",
        r"(?i)^(colorado\s+rockies)$": "Colorado",
        r"(?i)^(detroit\s+tigers)$": "Detroit",
        r"(?i)^(milwaukee\s+brewers)$": "Milwaukee",
        r"(?i)^(minnesota\s+twins)$": "Minnesota",
        r"(?i)^(atlanta\s+braves)$": "Atlanta",
        r"(?i)^(baltimore\s+orioles)$": "Baltimore",
        r"(?i)^(boston\s+red\s+sox)$": "Boston",
        r"(?i)^(miami\s+marlins)$": "Miami",
    }
    for pat, repl in rules.items():
        if re.match(pat, s):
            return repl
    return s

def replace_away_home_with_abbr(df: pd.DataFrame, teams_col: str = "TEAMS") -> pd.DataFrame:
    """
    From a 'TEAMS' column like 'Away at Home', create away/home and REPLACE them
    with abbreviations using team_to_abbr_short. Returns a new DataFrame.
    """
    if teams_col not in df.columns:
        raise KeyError(f"'{teams_col}' column not found.")

    out = df.copy()

    # Parse '<away> at <home>' (case-insensitive, tolerant spacing)
    pairs = (out[teams_col].astype(str)
             .str.extract(r"^(.*?)\s+at\s+(.*)$", flags=re.IGNORECASE))
    away = pairs[0].str.strip().map(_normalize_team_label)
    home = pairs[1].str.strip().map(_normalize_team_label)

    # Map to abbreviations and REPLACE away_team/home_team with the abbrs
    out["away_team"] = away.map(team_to_abbr_short)
    out["home_team"] = home.map(team_to_abbr_short)

    return out


In [8]:
dh_25 = replace_away_home_with_abbr(dh_25)  
dh_24 = replace_away_home_with_abbr(dh_24)  
dh_23 = replace_away_home_with_abbr(dh_23)  
dh_22 = replace_away_home_with_abbr(dh_22)  

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2025-04-06,St. Louis at Boston,"BOS,\n5-4","BOS,\n18-7",STL,BOS
1,2025-04-20,Washington at Colorado,"COL,\n3-1","WSH,\n3-2",WSH,COL
2,2025-04-24,Colorado at Kansas City,"KC,\n7-4","KC,\n6-2",COL,KC
3,2025-04-26,Boston at Cleveland,"CLE,\n5-4","BOS,\n7-3",BOS,CLE
4,2025-04-26,Baltimore at Detroit,"DET,\n4-3","DET,\n6-2",BAL,DET
5,2025-04-27,Toronto at NY Yankees,"NYY,\n5-1","NYY,\n11-2",TOR,NYY
6,2025-04-30,St. Louis at Cincinnati,"STL,\n9-1","STL,\n6-0",STL,CIN
7,2025-05-04,NY Mets at St. Louis,"STL,\n6-5","STL,\n5-4",NYM,STL
8,2025-05-06,Cleveland at Washington,"WSH,\n10-9","CLE,\n9-1",CLE,WSH
9,2025-05-08,Detroit at Colorado,"DET,\n10-2","DET,\n11-1",DET,COL


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2024-04-04,Detroit at NY Mets,"NYM,\n2-1","DET,\n6-3",DET,NYM
1,2024-04-13,NY Yankees at Cleveland,"NYY,\n3-2","NYY,\n8-2",NYY,CLE
2,2024-04-13,Minnesota at Detroit,"MIN,\n4-1","MIN,\n11-5",MIN,DET
3,2024-04-17,Kansas City at Chicago Sox,"KC,\n4-2","CHW,\n2-1",KC,CWS
4,2024-04-20,Miami at Chicago Cubs,"CHC,\n5-3","MIA,\n3-2",MIA,CHC
5,2024-04-21,Seattle at Colorado,"SEA,\n10-2","COL,\n2-1",SEA,COL
6,2024-04-30,St. Louis at Detroit,"DET,\n11-6","STL,\n2-1",STL,DET
7,2024-05-08,Texas at Athletics,"ATH,\n9-4","TEX,\n12-11",TEX,ATH
8,2024-05-14,Washington at Chicago Sox,"WSH,\n6-3","CHW,\n4-0",WSH,CWS
9,2024-05-20,San Diego at Atlanta,"ATL,\n3-0","SD,\n6-5",SD,ATL


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2023-04-18,Philadelphia at Chicago Sox,"CHW,\n3-0","PHI,\n7-4",PHI,CWS
1,2023-04-18,Cleveland at Detroit,"DET,\n4-3","DET,\n1-0",CLE,DET
2,2023-04-22,Miami at Cleveland,"MIA,\n3-2","MIA,\n6-1",MIA,CLE
3,2023-04-29,Baltimore at Detroit,"BAL,\n6-4","DET,\n7-4",BAL,DET
4,2023-04-29,Pittsburgh at Washington,"PIT,\n16-1","PIT,\n6-3",PIT,WSH
5,2023-05-01,Atlanta at NY Mets,"NYM,\n5-3","ATL,\n9-8",ATL,NYM
6,2023-05-03,NY Mets at Detroit,"DET,\n8-1","DET,\n6-5",NYM,DET
7,2023-05-21,Cleveland at NY Mets,"NYM,\n2-1","NYM,\n5-4",CLE,NYM
8,2023-06-03,Tampa Bay at Boston,"TB,\n4-2","BOS,\n8-5",TB,BOS
9,2023-06-08,Chicago Sox at NY Yankees,"CHW,\n6-5","NYY,\n3-0",CWS,NYY


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2022-04-19,Arizona at Washington,"WSH,\n6-1","WSH,\n1-0",AZ,WSH
1,2022-04-19,San Francisco at NY Mets,"NYM,\n5-4","NYM,\n3-1",SF,NYM
2,2022-04-20,Chicago Sox at Cleveland,"CLE,\n11-1","CLE,\n2-1",CWS,CLE
3,2022-04-23,Colorado at Detroit,"DET,\n13-0","COL,\n3-2",COL,DET
4,2022-05-03,Atlanta at NY Mets,"NYM,\n5-4","NYM,\n3-0",ATL,NYM
5,2022-05-04,San Diego at Cleveland,"SD,\n5-4","CLE,\n6-5",SD,CLE
6,2022-05-04,Pittsburgh at Detroit,"DET,\n3-2","PIT,\n7-2",PIT,DET
7,2022-05-07,Toronto at Cleveland,"TOR,\n8-3","CLE,\n8-2",TOR,CLE
8,2022-05-07,LA Dodgers at Chicago Cubs,"LAD,\n7-0","LAD,\n6-2",LAD,CHC
9,2022-05-07,Pittsburgh at Cincinnati,"CIN,\n9-2","PIT,\n8-5",PIT,CIN


### Expanding Double Headers

In [9]:
def expand_double_headers(df):
    """
    Take a double-header DataFrame (with GAME 1 / GAME 2 columns)
    and return a long DataFrame with one row per game.

    Columns required:
      - 'DATE'
      - 'TEAMS'
      - 'away_team'
      - 'home_team'
      - 'GAME 1'
      - 'GAME 2'

    Output columns include:
      - DATE
      - TEAMS
      - away_team
      - home_team
      - game_label   (GAME 1 / GAME 2)
      - raw_result   (e.g., 'BOS,\\n5-4')
      - game_number  (1 or 2)
    """
    required_cols = ["DATE", "TEAMS", "away_team", "home_team", "GAME 1", "GAME 2"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"expand_double_headers: missing required columns: {missing}")

    long_df = df.melt(
        id_vars=["DATE", "TEAMS", "away_team", "home_team"],
        value_vars=["GAME 1", "GAME 2"],
        var_name="game_label",
        value_name="raw_result"
    )

    # Extract game number: "GAME 1" -> 1, "GAME 2" -> 2
    long_df["game_number"] = long_df["game_label"].str.extract(r"(\d+)").astype(int)

    # Sort by date then game number
    long_df = (
        long_df
        .sort_values(["DATE", "game_number"])
        .reset_index(drop=True)
    )

    return long_df

In [10]:
dh_25 = expand_double_headers(dh_25)
dh_24 = expand_double_headers(dh_24)
dh_23 = expand_double_headers(dh_23)
dh_22 = expand_double_headers(dh_22)

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2025-04-06,St. Louis at Boston,STL,BOS,GAME 1,"BOS,\n5-4",1
1,2025-04-06,St. Louis at Boston,STL,BOS,GAME 2,"BOS,\n18-7",2
2,2025-04-20,Washington at Colorado,WSH,COL,GAME 1,"COL,\n3-1",1
3,2025-04-20,Washington at Colorado,WSH,COL,GAME 2,"WSH,\n3-2",2
4,2025-04-24,Colorado at Kansas City,COL,KC,GAME 1,"KC,\n7-4",1
5,2025-04-24,Colorado at Kansas City,COL,KC,GAME 2,"KC,\n6-2",2
6,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 1,"CLE,\n5-4",1
7,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 1,"DET,\n4-3",1
8,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 2,"BOS,\n7-3",2
9,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n6-2",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 1,"NYM,\n2-1",1
1,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 2,"DET,\n6-3",2
2,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 1,"NYY,\n3-2",1
3,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 1,"MIN,\n4-1",1
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 2,"NYY,\n8-2",2
5,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 2,"MIN,\n11-5",2
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 1,"KC,\n4-2",1
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 2,"CHW,\n2-1",2
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 1,"CHC,\n5-3",1
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 2,"MIA,\n3-2",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 1,"CHW,\n3-0",1
1,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 1,"DET,\n4-3",1
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 2,"PHI,\n7-4",2
3,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 2,"DET,\n1-0",2
4,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 1,"MIA,\n3-2",1
5,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 2,"MIA,\n6-1",2
6,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 1,"BAL,\n6-4",1
7,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 1,"PIT,\n16-1",1
8,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n7-4",2
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 2,"PIT,\n6-3",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2022-04-19,Arizona at Washington,AZ,WSH,GAME 1,"WSH,\n6-1",1
1,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 1,"NYM,\n5-4",1
2,2022-04-19,Arizona at Washington,AZ,WSH,GAME 2,"WSH,\n1-0",2
3,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 2,"NYM,\n3-1",2
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 1,"CLE,\n11-1",1
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 2,"CLE,\n2-1",2
6,2022-04-23,Colorado at Detroit,COL,DET,GAME 1,"DET,\n13-0",1
7,2022-04-23,Colorado at Detroit,COL,DET,GAME 2,"COL,\n3-2",2
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 1,"NYM,\n5-4",1
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 2,"NYM,\n3-0",2


### Creating Home and Away Scores

In [11]:
def add_home_away_scores(df):
    """
    Parse raw_result (e.g. 'BOS,\\n5-4') into integer home_score and away_score.

    Expects columns:
      - 'away_team'
      - 'home_team'
      - 'raw_result'

    Returns:
      DataFrame with:
        - home_score (Int64)
        - away_score (Int64)
    """
    def parse_row(row):
        raw = row["raw_result"]

        if pd.isna(raw):
            return pd.Series([np.nan, np.nan])

        try:
            # Split: "BOS,\n5-4" → ("BOS", "5-4")
            team_part, score_part = raw.split(",", 1)
            winner_team = team_part.strip()

            # Clean score and split → (5, 4)
            score_part = score_part.replace("\n", "").strip()
            runs_winner, runs_loser = map(int, score_part.split("-"))

            # Assign to home/away
            if winner_team == row["home_team"]:
                home_score = runs_winner
                away_score = runs_loser
            else:
                home_score = runs_loser
                away_score = runs_winner

        except Exception:
            home_score = np.nan
            away_score = np.nan

        return pd.Series([home_score, away_score])

    df[["home_score", "away_score"]] = df.apply(parse_row, axis=1)

    # Cast to nullable integer type
    df["home_score"] = df["home_score"].astype("Int64")
    df["away_score"] = df["away_score"].astype("Int64")

    return df


In [12]:
dh_25 = add_home_away_scores(dh_25)
dh_24 = add_home_away_scores(dh_24)
dh_23 = add_home_away_scores(dh_23)
dh_22 = add_home_away_scores(dh_22)

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,home_score,away_score
0,2025-04-06,St. Louis at Boston,STL,BOS,GAME 1,"BOS,\n5-4",1,5,4
1,2025-04-06,St. Louis at Boston,STL,BOS,GAME 2,"BOS,\n18-7",2,18,7
2,2025-04-20,Washington at Colorado,WSH,COL,GAME 1,"COL,\n3-1",1,3,1
3,2025-04-20,Washington at Colorado,WSH,COL,GAME 2,"WSH,\n3-2",2,2,3
4,2025-04-24,Colorado at Kansas City,COL,KC,GAME 1,"KC,\n7-4",1,7,4
5,2025-04-24,Colorado at Kansas City,COL,KC,GAME 2,"KC,\n6-2",2,6,2
6,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 1,"CLE,\n5-4",1,5,4
7,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 1,"DET,\n4-3",1,4,3
8,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 2,"BOS,\n7-3",2,3,7
9,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n6-2",2,6,2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,home_score,away_score
0,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 1,"NYM,\n2-1",1,2,1
1,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 2,"DET,\n6-3",2,3,6
2,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 1,"NYY,\n3-2",1,2,3
3,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 1,"MIN,\n4-1",1,1,4
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 2,"NYY,\n8-2",2,2,8
5,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 2,"MIN,\n11-5",2,5,11
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 1,"KC,\n4-2",1,2,4
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 2,"CHW,\n2-1",2,1,2
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 1,"CHC,\n5-3",1,5,3
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 2,"MIA,\n3-2",2,2,3


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,home_score,away_score
0,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 1,"CHW,\n3-0",1,0,3
1,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 1,"DET,\n4-3",1,4,3
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 2,"PHI,\n7-4",2,4,7
3,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 2,"DET,\n1-0",2,1,0
4,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 1,"MIA,\n3-2",1,2,3
5,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 2,"MIA,\n6-1",2,1,6
6,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 1,"BAL,\n6-4",1,4,6
7,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 1,"PIT,\n16-1",1,1,16
8,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n7-4",2,7,4
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 2,"PIT,\n6-3",2,3,6


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,home_score,away_score
0,2022-04-19,Arizona at Washington,AZ,WSH,GAME 1,"WSH,\n6-1",1,6,1
1,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 1,"NYM,\n5-4",1,5,4
2,2022-04-19,Arizona at Washington,AZ,WSH,GAME 2,"WSH,\n1-0",2,1,0
3,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 2,"NYM,\n3-1",2,3,1
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 1,"CLE,\n11-1",1,11,1
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 2,"CLE,\n2-1",2,2,1
6,2022-04-23,Colorado at Detroit,COL,DET,GAME 1,"DET,\n13-0",1,13,0
7,2022-04-23,Colorado at Detroit,COL,DET,GAME 2,"COL,\n3-2",2,2,3
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 1,"NYM,\n5-4",1,5,4
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 2,"NYM,\n3-0",2,3,0


## MLB Stats API

### Abbreviating Team Names

In [13]:
import requests
import pandas as pd
import numpy as np
from datetime import date
from collections import OrderedDict

# ---------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------
BASE = "https://statsapi.mlb.com"

TEAM_TO_ABBR = {
    "Arizona Diamondbacks": "AZ",
    "Oakland Athletics": "ATH",
    "Atlanta Braves": "ATL",
    "Baltimore Orioles": "BAL",
    "Boston Red Sox": "BOS",
    "Chicago Cubs": "CHC",
    "Chicago White Sox": "CWS",
    "Cincinnati Reds": "CIN",
    "Cleveland Guardians": "CLE",
    "Colorado Rockies": "COL",
    "Detroit Tigers": "DET",
    "Houston Astros": "HOU",
    "Kansas City Royals": "KC",
    "Los Angeles Angels": "LAA",
    "Los Angeles Dodgers": "LAD",
    "Miami Marlins": "MIA",
    "Milwaukee Brewers": "MIL",
    "Minnesota Twins": "MIN",
    "New York Mets": "NYM",
    "New York Yankees": "NYY",
    "Philadelphia Phillies": "PHI",
    "Pittsburgh Pirates": "PIT",
    "San Diego Padres": "SD",
    "San Francisco Giants": "SF",
    "Seattle Mariners": "SEA",
    "St. Louis Cardinals": "STL",
    "Tampa Bay Rays": "TB",
    "Texas Rangers": "TEX",
    "Toronto Blue Jays": "TOR",
    "Washington Nationals": "WSH",
}

# ---------------------------------------------------------------------
# Core MLB API helpers
# ---------------------------------------------------------------------
def fetch_live_feed(game_pk: int) -> dict:
    r = requests.get(f"{BASE}/api/v1.1/game/{game_pk}/feed/live")
    r.raise_for_status()
    return r.json()

def _name_lookup(feed):
    players = (feed.get("gameData", {}) or {}).get("players", {}) or {}
    return lambda pid: players.get(f"ID{pid}", {}).get("fullName") if pid else None

def _appearance_orders(feed):
    """
    Return (home_order, away_order) of pitcher IDs from play-by-play.
    Home defends TOP halves; Away defends BOTTOM halves.
    """
    plays = ((feed.get("liveData", {}) or {}).get("plays") or {}).get("allPlays") or []
    home_order, away_order = [], []
    last_top, last_bot = None, None

    for p in plays:
        half = (p.get("about") or {}).get("halfInning")
        pid  = (p.get("matchup") or {}).get("pitcher", {}).get("id")
        if not pid:
            continue
        if half == "top":  # home on defense
            if pid != last_top:
                home_order.append(pid)
                last_top = pid
        elif half == "bottom":  # away on defense
            if pid != last_bot:
                away_order.append(pid)
                last_bot = pid

    # de-dup but preserve order
    home_order = list(OrderedDict.fromkeys(home_order))
    away_order = list(OrderedDict.fromkeys(away_order))
    return home_order, away_order

def summarize_game_all_pitchers(feed: dict) -> dict:
    """
    Date/status/teams/score + ALL pitchers for each team (appearance-ordered).
    """
    name = _name_lookup(feed)
    gd   = feed.get("gameData", {}) or {}
    live = feed.get("liveData", {}) or {}

    game_date = (gd.get("datetime", {}) or {}).get("officialDate")
    status    = (gd.get("status", {}) or {}).get("detailedState")
    away_team = (gd.get("teams", {}).get("away", {}) or {}).get("name")
    home_team = (gd.get("teams", {}).get("home", {}) or {}).get("name")

    teams_ls  = (live.get("linescore", {}) or {}).get("teams", {}) or {}
    away_runs = (teams_ls.get("away", {}) or {}).get("runs")
    home_runs = (teams_ls.get("home", {}) or {}).get("runs")

    # appearance orders from plays
    home_order, away_order = _appearance_orders(feed)

    # Fallback if no plays yet (e.g., pregame): use boxscore order
    if not home_order or not away_order:
        box_teams = (live.get("boxscore", {}) or {}).get("teams", {}) or {}
        if not home_order:
            home_order = (box_teams.get("home", {}) or {}).get("pitchers") or []
        if not away_order:
            away_order = (box_teams.get("away", {}) or {}).get("pitchers") or []

    pitchers_home = [
        {"id": pid, "name": name(pid), "appearance_index": i}
        for i, pid in enumerate(home_order)
    ]
    pitchers_away = [
        {"id": pid, "name": name(pid), "appearance_index": i}
        for i, pid in enumerate(away_order)
    ]

    return {
        "date": game_date,
        "status": status,
        "away_team": away_team,
        "home_team": home_team,
        "away_runs": away_runs,
        "home_runs": home_runs,
        "pitchers_away": pitchers_away,
        "pitchers_home": pitchers_home,
    }

def get_game_pks(date_str=None, sport_id=1, team_id=None):
    if date_str is None:
        date_str = date.today().strftime("%Y-%m-%d")
    r = requests.get(f"{BASE}/api/v1/schedule", params={"sportId": sport_id, "date": date_str})
    r.raise_for_status()
    data = r.json()
    games = [g for d in data.get("dates", []) for g in d.get("games", [])]
    if team_id:
        games = [g for g in games
                 if g["teams"]["away"]["team"]["id"] == team_id
                 or g["teams"]["home"]["team"]["id"] == team_id]
    return [g["gamePk"] for g in games]

def summarize_date_all_pitchers(date_str=None, team_id=None):
    rows = []
    for pk in get_game_pks(date_str, team_id=team_id):
        feed = fetch_live_feed(pk)
        row = summarize_game_all_pitchers(feed)
        row["gamePk"] = pk
        rows.append(row)
    return rows

# ---------------------------------------------------------------------
# Team-name → abbreviation utilities
# ---------------------------------------------------------------------
def _abbr(name: str | None) -> str | None:
    return TEAM_TO_ABBR.get(name) if name else None

def add_team_abbr_to_summary(summary: dict, replace: bool = False) -> dict:
    """
    Given one game's summary dict (from summarize_game_all_pitchers),
    add away_team_abbr / home_team_abbr (or replace names if replace=True).
    """
    away = summary.get("away_team")
    home = summary.get("home_team")
    away_abbr = _abbr(away)
    home_abbr = _abbr(home)

    summary = dict(summary)  # shallow copy
    if replace:
        summary["away_team"] = away_abbr
        summary["home_team"] = home_abbr
    else:
        summary["away_team_abbr"] = away_abbr
        summary["home_team_abbr"] = home_abbr
    return summary

def add_team_abbr_to_rows(rows: list[dict], replace: bool = False) -> list[dict]:
    """Apply the mapping to a list of game summaries."""
    return [add_team_abbr_to_summary(r, replace=replace) for r in rows]

# ---------------------------------------------------------------------
# Per-date summary → DataFrame (with abbreviations)
# ---------------------------------------------------------------------
def summarize_date_all_pitchers_df(date_str=None, team_id=None, use_abbr: bool = True) -> pd.DataFrame:
    """
    Wrap summarize_date_all_pitchers into a tidy DataFrame for a given date.

    Returns columns:
      - date
      - status
      - away_team
      - home_team
      - away_runs (Int64)
      - home_runs (Int64)
      - pitchers_away
      - pitchers_home
      - gamePk
    """
    rows = summarize_date_all_pitchers(date_str=date_str, team_id=team_id)

    if use_abbr:
        # Replace full names with your abbreviations (MIA, WSH, etc.)
        rows = add_team_abbr_to_rows(rows, replace=True)

    df = pd.DataFrame(rows)

    # Make sure run columns are nullable integers
    for col in ["away_runs", "home_runs"]:
        if col in df.columns:
            df[col] = df[col].astype("Int64")

    return df

# ---------------------------------------------------------------------
# Main function: pitcher DF for double-header games
# ---------------------------------------------------------------------
def get_doubleheader_pitcher_df(dh_df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a double-header game DataFrame with columns:
      - 'DATE'        (date of game)
      - 'away_team'   (abbr, e.g. 'MIA')
      - 'home_team'   (abbr, e.g. 'WSH')
      - 'away_score'  (int)
      - 'home_score'  (int)
      - 'game_number' (1 for GAME 1, 2 for GAME 2)

    Fetch pitcher data from MLB StatsAPI for those dates and games,
    and return a DataFrame with one row per double-header game:

      - date          (original DATE from dh_df)
      - game_number   (1 or 2)
      - status        (e.g. 'Final')
      - away_team     (abbr)
      - home_team     (abbr)
      - away_runs
      - home_runs
      - pitchers_away (list of dicts)
      - pitchers_home (list of dicts)
      - gamePk
    """
    dh = dh_df.copy()

    if "game_number" not in dh.columns:
        raise KeyError("get_doubleheader_pitcher_df expects a 'game_number' column.")

    # Normalize DATE for merging with API results
    dt = pd.to_datetime(dh["DATE"])
    dh["DATE_str"] = dt.dt.strftime("%Y-%m-%d")

    # Scores as nullable ints
    dh["away_score"] = dh["away_score"].astype("Int64")
    dh["home_score"] = dh["home_score"].astype("Int64")

    # Unique dates we need to hit the API for
    unique_dates = sorted(dh["DATE_str"].unique())

    api_frames = []
    for d in unique_dates:
        api_df = summarize_date_all_pitchers_df(d, use_abbr=True)
        if not api_df.empty:
            api_df["DATE_str"] = d
            api_frames.append(api_df)

    if not api_frames:
        return pd.DataFrame(columns=[
            "date", "game_number", "status",
            "away_team", "home_team",
            "away_runs", "home_runs",
            "pitchers_away", "pitchers_home", "gamePk"
        ])

    all_api = pd.concat(api_frames, ignore_index=True)

    # Ensure runs are nullable ints
    all_api["away_runs"] = all_api["away_runs"].astype("Int64")
    all_api["home_runs"] = all_api["home_runs"].astype("Int64")

    # Merge DH games with MLB summaries on date + team + score
    merged = dh.merge(
        all_api[[
            "DATE_str", "away_team", "home_team",
            "away_runs", "home_runs", "status",
            "pitchers_away", "pitchers_home", "gamePk"
        ]],
        how="left",
        left_on=["DATE_str", "away_team", "home_team", "away_score", "home_score"],
        right_on=["DATE_str", "away_team", "home_team", "away_runs", "home_runs"],
    )

    # Final tidy shape
    result = merged[[
        "DATE",         # original date
        "game_number",
        "status",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "pitchers_away",
        "pitchers_home",
        "gamePk"
    ]].rename(columns={
        "DATE": "date",
        "away_score": "away_runs",
        "home_score": "home_runs",
    })

    # Sort by date and game_number so GAME 1 always comes before GAME 2
    result = result.sort_values(["date", "game_number"]).reset_index(drop=True)

    return result

In [14]:
dh_25_full = get_doubleheader_pitcher_df(dh_25)
dh_24_full = get_doubleheader_pitcher_df(dh_24)
dh_23_full = get_doubleheader_pitcher_df(dh_23)
dh_22_full = get_doubleheader_pitcher_df(dh_22)

display(HTML("<h4>Season 2025</h4>")); display(dh_25_full.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24_full.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23_full.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22_full.head(10))

,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2025-04-06,1,Final,STL,BOS,4,5,"[{'id': 669467, 'name': 'Andre Pallante', 'app...","[{'id': 656794, 'name': 'Sean Newcomb', 'appea...",778443.0
1,2025-04-06,2,Final,STL,BOS,7,18,"[{'id': 571945, 'name': 'Miles Mikolas', 'appe...","[{'id': 690928, 'name': 'Hunter Dobbins', 'app...",778432.0
2,2025-04-20,1,Final,WSH,COL,1,3,"[{'id': 695418, 'name': 'Brad Lord', 'appearan...","[{'id': 622608, 'name': 'Antonio Senzatela', '...",778266.0
3,2025-04-20,2,Final,WSH,COL,3,2,"[{'id': 663623, 'name': 'Jake Irvin', 'appeara...","[{'id': 607536, 'name': 'Kyle Freeland', 'appe...",778235.0
4,2025-04-24,1,Final,COL,KC,4,7,"[{'id': 608566, 'name': 'Germán Márquez', 'app...","[{'id': 666142, 'name': 'Cole Ragans', 'appear...",778190.0
5,2025-04-24,2,Final,COL,KC,2,6,"[{'id': 801403, 'name': 'Chase Dollander', 'ap...","[{'id': 547179, 'name': 'Michael Lorenzen', 'a...",778195.0
6,2025-04-26,1,Final,BOS,CLE,4,5,"[{'id': 656557, 'name': 'Tanner Houck', 'appea...","[{'id': 594902, 'name': 'Ben Lively', 'appeara...",778179.0
7,2025-04-26,1,Final,BAL,DET,3,4,"[{'id': 687064, 'name': 'Brandon Young', 'appe...","[{'id': 663554, 'name': 'Casey Mize', 'appeara...",778168.0
8,2025-04-26,2,Final,BOS,CLE,7,3,"[{'id': 621111, 'name': 'Walker Buehler', 'app...","[{'id': 680951, 'name': 'Doug Nikhazy', 'appea...",778169.0
9,2025-04-26,2,Final,BAL,DET,2,6,"[{'id': 669211, 'name': 'Keegan Akin', 'appear...","[{'id': 672456, 'name': 'Keider Montero', 'app...",778180.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2024-04-04,1,Final,DET,NYM,1,2,"[{'id': 666159, 'name': 'Matt Manning', 'appea...","[{'id': 676130, 'name': 'José Buttó', 'appeara...",745843.0
1,2024-04-04,2,Final,DET,NYM,6,3,"[{'id': 663554, 'name': 'Casey Mize', 'appeara...","[{'id': 605288, 'name': 'Adrian Houser', 'appe...",745844.0
2,2024-04-13,1,Final,NYY,CLE,3,2,"[{'id': 657376, 'name': 'Clarke Schmidt', 'app...","[{'id': 471911, 'name': 'Carlos Carrasco', 'ap...",746656.0
3,2024-04-13,1,Final,MIN,DET,4,1,"[{'id': 680573, 'name': 'Simeon Woods Richards...","[{'id': 666159, 'name': 'Matt Manning', 'appea...",746490.0
4,2024-04-13,2,Final,NYY,CLE,8,2,"[{'id': 547001, 'name': 'Cody Poteet', 'appear...","[{'id': 663474, 'name': 'Triston McKenzie', 'a...",746658.0
5,2024-04-13,2,Final,MIN,DET,11,5,"[{'id': 657746, 'name': 'Joe Ryan', 'appearanc...","[{'id': 628317, 'name': 'Kenta Maeda', 'appear...",746489.0
6,2024-04-17,1,Final,KC,CWS,4,2,"[{'id': 663903, 'name': 'Brady Singer', 'appea...","[{'id': 686563, 'name': 'Jonathan Cannon', 'ap...",746807.0
7,2024-04-17,2,NaN,KC,CWS,2,1,NaN,NaN,NaN
8,2024-04-20,1,Final,MIA,CHC,3,5,"[{'id': 682610, 'name': 'Roddery Muñoz', 'appe...","[{'id': 684007, 'name': 'Shota Imanaga', 'appe...",746891.0
9,2024-04-20,2,Final,MIA,CHC,3,2,"[{'id': 666200, 'name': 'Jesús Luzardo', 'appe...","[{'id': 665871, 'name': 'Javier Assad', 'appea...",746893.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2023-04-18,1,NaN,PHI,CWS,3,0,NaN,NaN,NaN
1,2023-04-18,1,Final,CLE,DET,3,4,"[{'id': 683769, 'name': 'Hunter Gaddis', 'appe...","[{'id': 571510, 'name': 'Matthew Boyd', 'appea...",718530.0
2,2023-04-18,2,Final,PHI,CWS,7,4,"[{'id': 554430, 'name': 'Zack Wheeler', 'appea...","[{'id': 458681, 'name': 'Lance Lynn', 'appeara...",718525.0
3,2023-04-18,2,Final,CLE,DET,0,1,"[{'id': 685410, 'name': 'Peyton Battenfield', ...","[{'id': 593958, 'name': 'Eduardo Rodriguez', '...",718541.0
4,2023-04-22,1,Final,MIA,CLE,3,2,"[{'id': 666129, 'name': 'Braxton Garrett', 'ap...","[{'id': 668676, 'name': 'Zach Plesac', 'appear...",718488.0
5,2023-04-22,2,Final,MIA,CLE,6,1,"[{'id': 656970, 'name': 'Devin Smeltzer', 'app...","[{'id': 669456, 'name': 'Shane Bieber', 'appea...",718473.0
6,2023-04-29,1,Final,BAL,DET,6,4,"[{'id': 680570, 'name': 'Grayson Rodriguez', '...","[{'id': 571510, 'name': 'Matthew Boyd', 'appea...",718401.0
7,2023-04-29,1,Final,PIT,WSH,16,1,"[{'id': 592826, 'name': 'Vince Velasquez', 'ap...","[{'id': 641771, 'name': 'Chad Kuhl', 'appearan...",718399.0
8,2023-04-29,2,Final,BAL,DET,4,7,"[{'id': 665152, 'name': 'Dean Kremer', 'appear...","[{'id': 593958, 'name': 'Eduardo Rodriguez', '...",718381.0
9,2023-04-29,2,Final,PIT,WSH,6,3,"[{'id': 448179, 'name': 'Rich Hill', 'appearan...","[{'id': 571578, 'name': 'Patrick Corbin', 'app...",718383.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2022-04-19,1,Final,AZ,WSH,1,6,"[{'id': 518516, 'name': 'Madison Bumgarner', '...","[{'id': 680686, 'name': 'Josiah Gray', 'appear...",662587.0
1,2022-04-19,1,Final,SF,NYM,4,5,"[{'id': 502171, 'name': 'Alex Cobb', 'appearan...","[{'id': 656731, 'name': 'Tylor Megill', 'appea...",662350.0
2,2022-04-19,2,Final,AZ,WSH,0,1,"[{'id': 656457, 'name': 'Tyler Gilbert', 'appe...","[{'id': 672851, 'name': 'Joan Adon', 'appearan...",662583.0
3,2022-04-19,2,Final,SF,NYM,1,3,"[{'id': 657277, 'name': 'Logan Webb', 'appeara...","[{'id': 453286, 'name': 'Max Scherzer', 'appea...",662616.0
4,2022-04-20,1,Final,CWS,CLE,1,11,"[{'id': 572971, 'name': 'Dallas Keuchel', 'app...","[{'id': 669456, 'name': 'Shane Bieber', 'appea...",663094.0
5,2022-04-20,2,Final,CWS,CLE,1,2,"[{'id': 669424, 'name': 'Jimmy Lambert', 'appe...","[{'id': 663474, 'name': 'Triston McKenzie', 'a...",663075.0
6,2022-04-23,1,Final,COL,DET,0,13,"[{'id': 622608, 'name': 'Antonio Senzatela', '...","[{'id': 669373, 'name': 'Tarik Skubal', 'appea...",662848.0
7,2022-04-23,2,Final,COL,DET,3,2,"[{'id': 596295, 'name': 'Austin Gomber', 'appe...","[{'id': 689225, 'name': 'Beau Brieske', 'appea...",662849.0
8,2022-05-03,1,Final,ATL,NYM,4,5,"[{'id': 450203, 'name': 'Charlie Morton', 'app...","[{'id': 656849, 'name': 'David Peterson', 'app...",662569.0
9,2022-05-03,2,Final,ATL,NYM,0,3,"[{'id': 657140, 'name': 'Kyle Wright', 'appear...","[{'id': 471911, 'name': 'Carlos Carrasco', 'ap...",662459.0


In [15]:
def pitcher_dicts_to_str(pitcher_list):
    """
    Convert a list of pitcher dicts like:
      [{'id': 669467, 'name': 'Andre Pallante', 'appearance_index': 0}, ...]
    into a comma-separated string of names: "Andre Pallante, ...".

    Safely handles None / NaN / non-lists.
    """
    if not isinstance(pitcher_list, list):
        return ""

    names = [
        p.get("name")
        for p in pitcher_list
        if isinstance(p, dict) and p.get("name")
    ]
    return ", ".join(names)


In [16]:
dh_25_full["pitchers_away"] = dh_25_full["pitchers_away"].apply(pitcher_dicts_to_str)
dh_25_full["pitchers_home"] = dh_25_full["pitchers_home"].apply(pitcher_dicts_to_str)

dh_24_full["pitchers_away"] = dh_24_full["pitchers_away"].apply(pitcher_dicts_to_str)
dh_24_full["pitchers_home"] = dh_24_full["pitchers_home"].apply(pitcher_dicts_to_str)

dh_23_full["pitchers_away"] = dh_23_full["pitchers_away"].apply(pitcher_dicts_to_str)
dh_23_full["pitchers_home"] = dh_23_full["pitchers_home"].apply(pitcher_dicts_to_str)

dh_22_full["pitchers_away"] = dh_22_full["pitchers_away"].apply(pitcher_dicts_to_str)
dh_22_full["pitchers_home"] = dh_22_full["pitchers_home"].apply(pitcher_dicts_to_str)


display(HTML("<h4>Season 2025</h4>")); display(dh_25_full.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24_full.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23_full.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22_full.head(10))

,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2025-04-06,1,Final,STL,BOS,4,5,"Andre Pallante, Kyle Leahy, JoJo Romero, Phil ...","Sean Newcomb, Greg Weissert, Justin Wilson, Ju...",778443.0
1,2025-04-06,2,Final,STL,BOS,7,18,"Miles Mikolas, Gordon Graceffo, John King, Chr...","Hunter Dobbins, Brennan Bernardino, Cooper Cri...",778432.0
2,2025-04-20,1,Final,WSH,COL,1,3,"Brad Lord, Colin Poche, Jackson Rutledge, Edua...","Antonio Senzatela, Jake Bird, Zach Agnos, Tyle...",778266.0
3,2025-04-20,2,Final,WSH,COL,3,2,"Jake Irvin, Jose A. Ferrer, Kyle Finnegan","Kyle Freeland, Angel Chivilli, Jimmy Herget, S...",778235.0
4,2025-04-24,1,Final,COL,KC,4,7,"Germán Márquez, Jake Bird, Jimmy Herget","Cole Ragans, Angel Zerpa, Steven Cruz, Lucas E...",778190.0
5,2025-04-24,2,Final,COL,KC,2,6,"Chase Dollander, Jaden Hill, Zach Agnos, Juan ...","Michael Lorenzen, John Schreiber, Daniel Lynch IV",778195.0
6,2025-04-26,1,Final,BOS,CLE,4,5,"Tanner Houck, Brennan Bernardino, Greg Weisser...","Ben Lively, Tim Herrin, Hunter Gaddis, Emmanue...",778179.0
7,2025-04-26,1,Final,BAL,DET,3,4,"Brandon Young, Bryan Baker, Cionel Pérez, Matt...","Casey Mize, Brenan Hanifee, Tyler Holton, Will...",778168.0
8,2025-04-26,2,Final,BOS,CLE,7,3,"Walker Buehler, Justin Wilson, Justin Slaten, ...","Doug Nikhazy, Kolby Allard",778169.0
9,2025-04-26,2,Final,BAL,DET,2,6,"Keegan Akin, Charlie Morton, Seranthony Domíng...","Keider Montero, Brant Hurter, Sean Guenther, C...",778180.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2024-04-04,1,Final,DET,NYM,1,2,"Matt Manning, Tyler Holton, Alex Faedo","José Buttó, Reed Garrett",745843.0
1,2024-04-04,2,Final,DET,NYM,6,3,"Casey Mize, Joey Wentz, Alex Lange, Andrew Cha...","Adrian Houser, Brooks Raley, Drew Smith, Jake ...",745844.0
2,2024-04-13,1,Final,NYY,CLE,3,2,"Clarke Schmidt, Caleb Ferguson, Ian Hamilton, ...","Carlos Carrasco, Nick Sandlin, Eli Morgan, Tim...",746656.0
3,2024-04-13,1,Final,MIN,DET,4,1,"Simeon Woods Richardson, Kody Funderburk, Cole...","Matt Manning, Joey Wentz",746490.0
4,2024-04-13,2,Final,NYY,CLE,8,2,"Cody Poteet, Dennis Santana, Ron Marinaccio","Triston McKenzie, Tyler Beede, Cade Smith, Wes...",746658.0
5,2024-04-13,2,Final,MIN,DET,11,5,"Joe Ryan, Steven Okert, Griffin Jax, Brock Ste...","Kenta Maeda, Tyler Holton, Shelby Miller, Jaso...",746489.0
6,2024-04-17,1,Final,KC,CWS,4,2,"Brady Singer, Will Smith, Nick Anderson, John ...","Jonathan Cannon, Jordan Leasure, Steven Wilson...",746807.0
7,2024-04-17,2,NaN,KC,CWS,2,1,,,NaN
8,2024-04-20,1,Final,MIA,CHC,3,5,"Roddery Muñoz, Anthony Bender, Bryan Hoeing, S...","Shota Imanaga, Ben Brown, Héctor Neris",746891.0
9,2024-04-20,2,Final,MIA,CHC,3,2,"Jesús Luzardo, Calvin Faucher, Tanner Scott","Javier Assad, Luke Little, Yency Almonte, Mark...",746893.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2023-04-18,1,NaN,PHI,CWS,3,0,,,NaN
1,2023-04-18,1,Final,CLE,DET,3,4,"Hunter Gaddis, Eli Morgan, Nick Sandlin, James...","Matthew Boyd, Mason Englert, Alex Lange",718530.0
2,2023-04-18,2,Final,PHI,CWS,7,4,"Zack Wheeler, Gregory Soto, Craig Kimbrel, Ser...","Lance Lynn, Jimmy Lambert, Gregory Santos, Jak...",718525.0
3,2023-04-18,2,Final,CLE,DET,0,1,"Peyton Battenfield, Xzavion Curry","Eduardo Rodriguez, Jason Foley",718541.0
4,2023-04-22,1,Final,MIA,CLE,3,2,"Braxton Garrett, Tanner Scott, Dylan Floro, A....","Zach Plesac, James Karinchak, Eli Morgan, Emma...",718488.0
5,2023-04-22,2,Final,MIA,CLE,6,1,"Devin Smeltzer, Andrew Nardi, Huascar Brazobán...","Shane Bieber, Nick Sandlin, Tim Herrin, Enyel ...",718473.0
6,2023-04-29,1,Final,BAL,DET,6,4,"Grayson Rodriguez, Keegan Akin, Mike Baumann, ...","Matthew Boyd, José Cisnero, Jason Foley, Will ...",718401.0
7,2023-04-29,1,Final,PIT,WSH,16,1,"Vince Velasquez, Cody Bolton, Yohan Ramírez","Chad Kuhl, Jordan Weems, Hobie Harris, Hunter ...",718399.0
8,2023-04-29,2,Final,BAL,DET,4,7,"Dean Kremer, DL Hall","Eduardo Rodriguez, Mason Englert, Alex Lange",718381.0
9,2023-04-29,2,Final,PIT,WSH,6,3,"Rich Hill, Robert Stephenson, Colin Holderman,...","Patrick Corbin, Carl Edwards Jr., Thaddeus War...",718383.0


,date,game_number,status,away_team,home_team,away_runs,home_runs,pitchers_away,pitchers_home,gamePk
0,2022-04-19,1,Final,AZ,WSH,1,6,"Madison Bumgarner, J.B. Wendelken, Oliver Pére...","Josiah Gray, Sean Doolittle, Steve Cishek, Kyl...",662587.0
1,2022-04-19,1,Final,SF,NYM,4,5,"Alex Cobb, Dominic Leone, Jose Alvarez, Jake M...","Tylor Megill, Joely Rodríguez, Seth Lugo, Edwi...",662350.0
2,2022-04-19,2,Final,AZ,WSH,0,1,"Tyler Gilbert, Sean Poppen, Joe Mantiply","Joan Adon, Víctor Arano, Kyle Finnegan, Tanner...",662583.0
3,2022-04-19,2,Final,SF,NYM,1,3,"Logan Webb, Sam Long, Zack Littell, John Brebb...","Max Scherzer, Drew Smith, Trevor May",662616.0
4,2022-04-20,1,Final,CWS,CLE,1,11,"Dallas Keuchel, Tanner Banks, Matt Foster, And...","Shane Bieber, Bryan Shaw, Enyel De Los Santos,...",663094.0
5,2022-04-20,2,Final,CWS,CLE,1,2,"Jimmy Lambert, Reynaldo López, Bennett Sousa, ...","Triston McKenzie, Anthony Gose, Nick Sandlin, ...",663075.0
6,2022-04-23,1,Final,COL,DET,0,13,"Antonio Senzatela, Ty Blach, Lucas Gilbreath, ...","Tarik Skubal, Wily Peralta, Angel De Jesus",662848.0
7,2022-04-23,2,Final,COL,DET,3,2,"Austin Gomber, Robert Stephenson, Tyler Kinley...","Beau Brieske, Alex Lange, Will Vest, Drew Hutc...",662849.0
8,2022-05-03,1,Final,ATL,NYM,4,5,"Charlie Morton, Jesse Chavez","David Peterson, Adam Ottavino, Drew Smith, Edw...",662569.0
9,2022-05-03,2,Final,ATL,NYM,0,3,"Kyle Wright, Will Smith","Carlos Carrasco, Seth Lugo",662459.0


## Exporting Data

In [17]:
# Collect existing double-header DataFrames into a dict automatically
double_headers = {
    year: df for year in range(2022, 2026)
    if (df := globals().get(f"dh_{str(year)[-2:]}_full")) is not None
}

# Create subfolder: data/double_headers
outdir = Path("data") / "double_headers"
outdir.mkdir(parents=True, exist_ok=True)

for year, df in double_headers.items():
    df.to_csv(outdir / f"double_headers_{year}.csv", index=False, encoding="utf-8")
    print(f"[saved] {outdir / f'double_headers_{year}.csv'}")


[saved] data/double_headers/double_headers_2022.csv
[saved] data/double_headers/double_headers_2023.csv
[saved] data/double_headers/double_headers_2024.csv
[saved] data/double_headers/double_headers_2025.csv
